In [1]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

In [2]:
import nbformat
from nbconvert import PythonExporter

notebook_path = '/content/drive/MyDrive/FWI/Forward_modelling.ipynb'
output_script_path = '/content/Forward_modelling.py'

with open(notebook_path, 'r', encoding='utf-8') as f:
    notebook_content = nbformat.read(f, as_version=4)
exporter = PythonExporter()
python_code, _ = exporter.from_notebook_node(notebook_content)
with open(output_script_path, 'w', encoding='utf-8') as f:
    f.write(python_code)
import sys
sys.path.append('/content/Forward_modelling')
import Forward_modelling

In [3]:
from Forward_modelling import helmholtz5
from Forward_modelling import helmholtzabc

In [4]:
# def Gradient(c_init, frequencies, d_obs,
#                         P, rec_idx,
#                         src_x, src_z,
#                         nx, nz, n, idx,
#                         dx,dz,npml,vmax,
#                         n_iterations):

#     c = c_init.copy()
#     misfits = []

#     for f in frequencies:

#         print("\n===== Frequency:", f, "Hz =====")
#         omega = 2*np.pi*f

#         for it in range(n_iterations):

#             g_c = np.zeros((nz, nx))
#             misfit = 0.0

#             # Model parameter
#             m = 1 / c**2
#             A = Forward_modelling.helmholtz5(m,omega,dx,dz,npml,vmax)
#             LU = spla.splu(A.tocsc())

#             # -------- Forward + Adjoint --------
#             for isrc, s in enumerate(src_x):

#                 q = np.zeros(n, dtype=complex)
#                 q[idx(src_z, s)] = 1.0

#                 # Forward
#                 u = LU.solve(q)

#                 # Residual
#                 r = P @ u - d_obs[f][isrc]
#                 misfit += 0.5 * np.linalg.norm(r)**2

#                 # Adjoint
#                 rhs = np.zeros(n, dtype=complex)
#                 rhs[rec_idx] = r
#                 lam = LU.solve(rhs, trans='H')

#                 # Gradient wrt m
#                 g_m = -omega**2 * np.real(u.conj() * lam).reshape(nz, nx)

#                 # Chain rule:  dm/dc = -2/c^3
#                 g_c += g_m * (-2 / (c**3))

#             misfits.append(misfit)
#             print("Iter:", it+1, "Misfit:", misfit)

#             # -------- Smoothing --------
#             g_c = gaussian_filter(g_c, sigma=2)

#             # -------- Normalization --------
#             g_c /= (np.max(np.abs(g_c)) + 1e-12)

#             # -------- Backtracking line search --------
#             step = 200.0
#             beta = 0.5

#             while step > 1e-6:

#                 c_trial = c - step * g_c
#                 c_trial = np.clip(c_trial, 1500, 2600)

#                 m_trial = 1 / c_trial**2
#                 A_trial = Forward_modelling.helmholtz5(m_trial,omega,dx,dz,npml,vmax)
#                 LU_trial = spla.splu(A_trial.tocsc())

#                 misfit_trial = 0.0

#                 for isrc, s in enumerate(src_x):
#                     q = np.zeros(n, dtype=complex)
#                     q[idx(src_z, s)] = 1.0
#                     u_trial = LU_trial.solve(q)
#                     r_trial = P @ u_trial - d_obs[f][isrc]
#                     misfit_trial += 0.5 * np.linalg.norm(r_trial)**2

#                 if misfit_trial < misfit:
#                     break

#                 step *= beta


#             c=c_trial
#             m = 1/c**2

#     return m, misfits

In [5]:
def Gradient(c_init, frequencies, d_obs,
             P, rec_idx,
             src_x, src_z,
             nx, nz, n, idx,
             dx, dz, npml, vmax,
             n_iterations):

    import numpy as np
    import scipy.sparse.linalg as spla
    from scipy.ndimage import gaussian_filter

    # ----------------------------------------
    # Parameterization: m = 1 / c^2
    # ----------------------------------------
    m = 1.0 / (c_init.copy() ** 2)

    # Bounds converted from velocity limits
    m_min = 1.0 / (2600.0 ** 2)
    m_max = 1.0 / (1500.0 ** 2)

    misfits = []

    for f in frequencies:

        print("\n===== Frequency:", f, "Hz =====")
        omega = 2 * np.pi * f

        for it in range(n_iterations):

            g_m_total = np.zeros((nz, nx))
            misfit = 0.0

            # Build operator directly with m
            A = Forward_modelling.helmholtz5(m, omega, dx, dz, npml, vmax)
            LU = spla.splu(A.tocsc())

            # -------- Forward + Adjoint --------
            for isrc, s in enumerate(src_x):

                q = np.zeros(n, dtype=complex)
                q[idx(src_z, s)] = 1.0

                # Forward
                u = LU.solve(q)

                # Residual
                r = P @ u - d_obs[f][isrc]
                misfit += 0.5 * np.linalg.norm(r)**2

                # Adjoint
                rhs = np.zeros(n, dtype=complex)
                rhs[rec_idx] = r
                lam = LU.solve(rhs, trans='H')

                # Gradient directly wrt squared slowness
                g_m = -omega**2 * np.real(u.conj() * lam).reshape(nz, nx)

                g_m_total += g_m

            misfits.append(misfit)
            print("Iter:", it+1, "Misfit:", misfit)

            # -------- Smoothing --------
            g_m_total = gaussian_filter(g_m_total, sigma=2)

            # -------- Normalization (kept identical to original code) --------
            g_m_total /= (np.max(np.abs(g_m_total)) + 1e-12)

            # -------- Backtracking line search --------
            step = 1e-6
            beta = 0.5

            while step > 1e-12:

                m_trial = m - step * g_m_total

                # Apply squared-slowness bounds
                m_trial = np.clip(m_trial, m_min, m_max)

                A_trial = Forward_modelling.helmholtz5(
                    m_trial, omega, dx, dz, npml, vmax
                )

                LU_trial = spla.splu(A_trial.tocsc())

                misfit_trial = 0.0

                for isrc, s in enumerate(src_x):

                    q = np.zeros(n, dtype=complex)
                    q[idx(src_z, s)] = 1.0

                    u_trial = LU_trial.solve(q)
                    r_trial = P @ u_trial - d_obs[f][isrc]
                    misfit_trial += 0.5 * np.linalg.norm(r_trial)**2

                if misfit_trial < misfit:
                    break

                step *= beta

            m = m_trial

    return m, misfits

In [6]:
def compute_gradient_field(m,
                           frequency,
                           d_obs,
                           P, rec_idx,
                           src_x, src_z,
                           nx, nz, n, idx,
                           dx, dz, npml, vmax):

    import numpy as np
    import scipy.sparse.linalg as spla

    omega = 2 * np.pi * frequency
    g_m_total = np.zeros((nz, nx))
    misfit = 0.0

    A = Forward_modelling.helmholtz5(m, omega, dx, dz, npml, vmax)
    LU = spla.splu(A.tocsc())

    for isrc, s in enumerate(src_x):

        q = np.zeros(n, dtype=complex)
        q[idx(src_z, s)] = 1.0

        u = LU.solve(q)

        r = P @ u - d_obs[frequency][isrc]
        misfit += 0.5 * np.linalg.norm(r)**2

        rhs = np.zeros(n, dtype=complex)
        rhs[rec_idx] = r
        lam = LU.solve(rhs, trans='H')

        g_m = -omega**2 * np.real(u.conj() * lam).reshape(nz, nx)

        g_m_total += g_m

    return g_m_total, misfit

In [7]:
def compute_forward_adjoint_fields(m,
                                   frequency,
                                   d_obs,
                                   P, rec_idx,
                                   src_x, src_z,
                                   nx, nz, n, idx,
                                   dx, dz, npml, vmax):

    import numpy as np
    import scipy.sparse.linalg as spla

    omega = 2 * np.pi * frequency

    A = Forward_modelling.helmholtz5(m, omega, dx, dz, npml, vmax)
    LU = spla.splu(A.tocsc())

    forward_fields = []
    adjoint_fields = []
    product_fields = []

    for isrc, s in enumerate(src_x):

        # -------- Forward --------
        q = np.zeros(n, dtype=complex)
        q[idx(src_z, s)] = 1.0
        u = LU.solve(q)

        # -------- Residual --------
        r = P @ u - d_obs[frequency][isrc]

        # -------- Adjoint --------
        rhs = np.zeros(n, dtype=complex)
        rhs[rec_idx] = r
        lam = LU.solve(rhs, trans='H')

        # -------- Reshape --------
        u2D = u.reshape(nz, nx)
        lam2D = lam.reshape(nz, nx)

        # -------- Product (core of gradient) --------
        product = np.real(u2D.conj() * lam2D)

        forward_fields.append(u2D)
        adjoint_fields.append(lam2D)
        product_fields.append(product)

    return forward_fields, adjoint_fields, product_fields

In [8]:
# def Gauss_newton(c_init, frequencies, d_obs,
#                          P, rec_idx,
#                         src_x, src_z,
#                         nx, nz, n, idx,
#                         dx,dz,npml,vmax,
#                         alpha,
#                         n_iterations,
#                         cg_maxiter):

#     m =  1/c_init**2
#     misfits = []

#     for f in frequencies:

#         print("\n===== Frequency:", f, "Hz =====")

#         omega = 2*np.pi*f
#         alpha =  alpha/(f**2)

#         for iteration in range(n_iterations):

#             g_total = np.zeros(n)
#             H_data = []
#             misfit = 0.0

#             # Forward operator
#             A = Forward_modelling.helmholtz5(m,omega,dx,dz,npml,vmax)
#             LU = spla.splu(A.tocsc())

#             # -------- Forward + Adjoint --------
#             for isrc, s in enumerate(src_x):

#                 q = np.zeros(n, dtype=complex)
#                 q[idx(src_z, s)] = 1.0

#                 # Forward
#                 u = LU.solve(q)

#                 # Residual
#                 r = P @ u - d_obs[f][isrc]
#                 misfit += 0.5 * np.linalg.norm(r)**2

#                 # Adjoint
#                 rhs = np.zeros(n, dtype=complex)
#                 rhs[rec_idx] = r
#                 lam = LU.solve(rhs, trans='H')

#                 # Gradient accumulation
#                 g_total += -omega**2 * np.real(u.conj() * lam)

#                 # Store for Gauss–Newton Hessian
#                 H_data.append((u, LU))

#             misfits.append(misfit)
#             print("Iteration:", iteration+1, "Misfit:", misfit)

#             # -------- Gauss–Newton Hessian-vector product --------
#             def H_matvec(v):

#                 Hv = np.zeros(n)

#                 for (u, LU_local) in H_data:

#                     w = LU_local.solve(-omega**2 * (v * u))
#                     Jv = P @ w

#                     rhs2 = np.zeros(n, dtype=complex)
#                     rhs2[rec_idx] = Jv
#                     lam2 = LU_local.solve(rhs2, trans='H')

#                     Hv += -omega**2 * np.real(u.conj() * lam2)

#                 Hv += alpha * v
#                 return Hv

#             H_linop = spla.LinearOperator((n, n), matvec=H_matvec)

#             # Solve H dm = -g  (CG)
#             dm, info = spla.cg(H_linop, -g_total, maxiter=cg_maxiter)

#             # -------- Line search --------
#             step = 1.0

#             for _ in range(8):

#                 m_trial = m + step * dm.reshape(nz, nx)

#                 A_trial =Forward_modelling.helmholtz5(m_trial,omega,dx,dz,npml,vmax)
#                 LU_trial = spla.splu(A_trial.tocsc())

#                 misfit_trial = 0.0

#                 for isrc, s in enumerate(src_x):
#                     q = np.zeros(n, dtype=complex)
#                     q[idx(src_z, s)] = 1.0
#                     u_trial = LU_trial.solve(q)
#                     r_trial = P @ u_trial - d_obs[f][isrc]
#                     misfit_trial += 0.5 * np.linalg.norm(r_trial)**2

#                 if misfit_trial < misfit:
#                     break

#                 step *= 0.5

#             # Update
#             m += step * dm.reshape(nz, nx)

#             # Mild stabilization
#             m = gaussian_filter(m, sigma=1)

#     return m, misfits

In [9]:
##### adaptive dapming
def Gauss_newton(c_init, frequencies, d_obs,
                 P, rec_idx,
                 src_x, src_z,
                 nx, nz, n, idx,
                 dx, dz, npml, vmax,
                 beta0,
                 n_iterations,
                 cg_maxiter):

    m = 1 / c_init**2
    misfits = []

    for f in frequencies:

        print("\n===== Frequency:", f, "Hz =====")

        omega = 2 * np.pi * f

        # fresh damping per frequency (scaled if desired)
        beta = beta0 / (f**2)

        for iteration in range(n_iterations):

            g_total = np.zeros(n)
            H_data = []
            misfit = 0.0

            A = Forward_modelling.helmholtz5(m, omega, dx, dz, npml, vmax)
            LU = spla.splu(A.tocsc())

            # -------- Forward + Adjoint --------
            for isrc, s in enumerate(src_x):

                q = np.zeros(n, dtype=complex)
                q[idx(src_z, s)] = 1.0

                u = LU.solve(q)

                r = P @ u - d_obs[f][isrc]
                misfit += 0.5 * np.linalg.norm(r)**2

                rhs = np.zeros(n, dtype=complex)
                rhs[rec_idx] = r
                lam = LU.solve(rhs, trans='H')

                g_total += -omega**2 * np.real(u.conj() * lam)

                H_data.append((u, LU))

            misfits.append(misfit)

            print("Iteration:", iteration+1,
                  "Misfit:", misfit,
                  "beta:", beta)

            # -------- Hessian-vector product --------
            def H_matvec(v):

                Hv = np.zeros(n)

                for (u, LU_local) in H_data:

                    w = LU_local.solve(-omega**2 * (v * u))
                    Jv = P @ w

                    rhs2 = np.zeros(n, dtype=complex)
                    rhs2[rec_idx] = Jv
                    lam2 = LU_local.solve(rhs2, trans='H')

                    Hv += -omega**2 * np.real(u.conj() * lam2)

                Hv += beta * v
                return Hv

            H_linop = spla.LinearOperator((n, n), matvec=H_matvec)

            dm, info = spla.cg(H_linop, -g_total, maxiter=cg_maxiter)

            # -------- Line search (FIXED) --------
            step = 1.0
            accepted = False

            for _ in range(8):

                m_trial = m + step * dm.reshape(nz, nx)

                A_trial = Forward_modelling.helmholtz5(
                    m_trial, omega, dx, dz, npml, vmax
                )
                LU_trial = spla.splu(A_trial.tocsc())

                misfit_trial = 0.0

                for isrc, s in enumerate(src_x):
                    q = np.zeros(n, dtype=complex)
                    q[idx(src_z, s)] = 1.0
                    u_trial = LU_trial.solve(q)
                    r_trial = P @ u_trial - d_obs[f][isrc]
                    misfit_trial += 0.5 * np.linalg.norm(r_trial)**2

                if misfit_trial < misfit:
                    accepted = True
                    break

                step *= 0.5

            # -------- LM damping update (FIXED LOCATION) --------
            if accepted:
                m = m_trial
                beta *= 0.5
            else:
                beta *= 5.0

            # Stabilization
            m = gaussian_filter(m, sigma=1)

    return m, misfits

In [10]:
# import numpy as np
# import scipy.sparse.linalg as spla

# def discrepancy_principle(m,
#                           beta_values,
#                           f,
#                           d_obs,
#                           P,
#                           rec_idx,
#                           src_x, src_z,
#                           nx, nz, n, idx,
#                           dx, dz, npml, vmax,
#                           noise_level,
#                           cg_maxiter):

#     omega = 2*np.pi*f

#     # ----- Build gradient + Hessian data -----
#     g_total = np.zeros(n)
#     H_data = []

#     A = Forward_modelling.helmholtz5(m, omega, dx, dz, npml, vmax)
#     LU = spla.splu(A.tocsc())

#     for isrc, s in enumerate(src_x):

#         q = np.zeros(n, dtype=complex)
#         q[idx(src_z, s)] = 1.0

#         u = LU.solve(q)

#         r = P @ u - d_obs[f][isrc]

#         rhs = np.zeros(n, dtype=complex)
#         rhs[rec_idx] = r
#         lam = LU.solve(rhs, trans='H')

#         g_total += -omega**2 * np.real(u.conj() * lam)
#         H_data.append((u, LU))

#     residual_norms = []

#     for beta in beta_values:

#         def H_matvec(v):
#             Hv = np.zeros(n)

#             for (u, LU_local) in H_data:
#                 w = LU_local.solve(-omega**2 * (v * u))
#                 Jv = P @ w

#                 rhs2 = np.zeros(n, dtype=complex)
#                 rhs2[rec_idx] = Jv
#                 lam2 = LU_local.solve(rhs2, trans='H')

#                 Hv += -omega**2 * np.real(u.conj() * lam2)

#             Hv += beta * v
#             return Hv

#         H_linop = spla.LinearOperator((n, n), matvec=H_matvec)

#         dm, _ = spla.cg(H_linop, -g_total, maxiter=cg_maxiter)

#         res_norm = np.linalg.norm(g_total + H_matvec(dm))
#         residual_norms.append(res_norm)

#     residual_norms = np.array(residual_norms)
#     idx_best = np.argmin(np.abs(residual_norms - noise_level))

#     return beta_values[idx_best], residual_norms

In [11]:


def discrepancy_principle(m,
                          beta_values,      # scaled beta (order 1)
                          f,
                          d_obs,
                          P,
                          rec_idx,
                          src_x, src_z,
                          nx, nz, n, idx,
                          dx, dz, npml, vmax,
                          noise_level,
                          cg_maxiter):

    omega = 2 * np.pi * f

    # --------------------------------------------------
    # Build gradient and store wavefields
    # --------------------------------------------------
    g_total = np.zeros(n)
    H_data = []

    A = Forward_modelling.helmholtz5(m, omega, dx, dz, npml, vmax)
    LU = spla.splu(A.tocsc())

    for isrc, s in enumerate(src_x):

        q = np.zeros(n, dtype=complex)
        q[idx(src_z, s)] = 1.0

        u = LU.solve(q)
        r = P @ u - d_obs[f][isrc]

        rhs = np.zeros(n, dtype=complex)
        rhs[rec_idx] = r
        lam = LU.solve(rhs, trans='H')

        g_total += -omega**2 * np.real(u.conj() * lam)

        # store forward field and LU
        H_data.append((u, LU))

    # --------------------------------------------------
    # Undamped Hessian operator
    # --------------------------------------------------
    def H0_matvec(v):

        Hv = np.zeros(n)

        for (u, LU_local) in H_data:

            w = LU_local.solve(-omega**2 * (v * u))
            Jv = P @ w

            rhs2 = np.zeros(n, dtype=complex)
            rhs2[rec_idx] = Jv
            lam2 = LU_local.solve(rhs2, trans='H')

            Hv += -omega**2 * np.real(u.conj() * lam2)

        return Hv

    # --------------------------------------------------
    # Compute effective spectral scale
    # --------------------------------------------------
    lambda_eff = g_total.dot(H0_matvec(g_total)) / \
                 (g_total.dot(g_total) + 1e-16)

    print("Effective eigenvalue scale:", lambda_eff)

    # --------------------------------------------------
    # Loop over scaled beta values
    # --------------------------------------------------
    residual_norms = []

    for beta in beta_values:

        beta_physical = beta * lambda_eff

        def H_matvec(v):

            Hv = H0_matvec(v)
            Hv += beta_physical * v
            return Hv

        H_linop = spla.LinearOperator((n, n), matvec=H_matvec)

        dm, _ = spla.cg(H_linop, -g_total, maxiter=cg_maxiter)

        res_norm = np.linalg.norm(g_total + H_matvec(dm))
        residual_norms.append(res_norm)

    residual_norms = np.array(residual_norms)

    idx_best = np.argmin(np.abs(residual_norms - noise_level))

    return beta_values[idx_best], residual_norms

In [12]:
# def jackson_principle(m,
#                       beta_values,
#                       f,
#                       d_obs,
#                       P,
#                       rec_idx,
#                       src_x, src_z,
#                       nx, nz, n, idx,
#                       dx, dz, npml, vmax,
#                       cg_maxiter):

#     omega = 2*np.pi*f

#     g_total = np.zeros(n)
#     H_data = []

#     A = Forward_modelling.helmholtz5(m, omega, dx, dz, npml, vmax)
#     LU = spla.splu(A.tocsc())

#     for isrc, s in enumerate(src_x):

#         q = np.zeros(n, dtype=complex)
#         q[idx(src_z, s)] = 1.0

#         u = LU.solve(q)
#         r = P @ u - d_obs[f][isrc]

#         rhs = np.zeros(n, dtype=complex)
#         rhs[rec_idx] = r
#         lam = LU.solve(rhs, trans='H')

#         g_total += -omega**2 * np.real(u.conj() * lam)
#         H_data.append((u, LU))

#     phi = []
#     psi = []

#     for beta in beta_values:

#         def H_matvec(v):
#             Hv = np.zeros(n)

#             for (u, LU_local) in H_data:
#                 w = LU_local.solve(-omega**2 * (v * u))
#                 Jv = P @ w

#                 rhs2 = np.zeros(n, dtype=complex)
#                 rhs2[rec_idx] = Jv
#                 lam2 = LU_local.solve(rhs2, trans='H')

#                 Hv += -omega**2 * np.real(u.conj() * lam2)

#             Hv += beta * v
#             return Hv

#         H_linop = spla.LinearOperator((n, n), matvec=H_matvec)

#         dm, _ = spla.cg(H_linop, -g_total, maxiter=cg_maxiter)

#         phi.append(np.linalg.norm(g_total + H_matvec(dm)))
#         psi.append(np.linalg.norm(dm))

#     phi = np.log(np.array(phi))
#     psi = np.log(np.array(psi))

#     dphi = np.gradient(phi)
#     dpsi = np.gradient(psi)
#     ddphi = np.gradient(dphi)
#     ddpsi = np.gradient(dpsi)

#     #curvature = np.abs(dphi * ddpsi - dpsi * ddphi) / \
#                 #(dphi**2 + dpsi**2)**1.5
#     eps = 1e-12

#     den = (dphi**2 + dpsi**2)**1.5
#     curvature = np.abs(dphi * ddpsi - dpsi * ddphi) / (den + eps)

#     curvature[np.isnan(curvature)] = 0
#     curvature[np.isinf(curvature)] = 0

#     idx_best = np.argmax(curvature)

#     return beta_values[idx_best], phi, psi, curvature

In [13]:


def jackson_principle(m,
                      beta_values,     # scaled beta (order 1)
                      f,
                      d_obs,
                      P,
                      rec_idx,
                      src_x, src_z,
                      nx, nz, n, idx,
                      dx, dz, npml, vmax,
                      cg_maxiter):

    omega = 2 * np.pi * f

    # --------------------------------------------------
    # Build gradient and store wavefields
    # --------------------------------------------------
    g_total = np.zeros(n)
    H_data = []

    A = Forward_modelling.helmholtz5(m, omega, dx, dz, npml, vmax)
    LU = spla.splu(A.tocsc())

    for isrc, s in enumerate(src_x):

        q = np.zeros(n, dtype=complex)
        q[idx(src_z, s)] = 1.0

        u = LU.solve(q)
        r = P @ u - d_obs[f][isrc]

        rhs = np.zeros(n, dtype=complex)
        rhs[rec_idx] = r
        lam = LU.solve(rhs, trans='H')

        g_total += -omega**2 * np.real(u.conj() * lam)

        H_data.append((u, LU))

    # --------------------------------------------------
    # Undamped Hessian operator
    # --------------------------------------------------
    def H0_matvec(v):

        Hv = np.zeros(n)

        for (u, LU_local) in H_data:

            w = LU_local.solve(-omega**2 * (v * u))
            Jv = P @ w

            rhs2 = np.zeros(n, dtype=complex)
            rhs2[rec_idx] = Jv
            lam2 = LU_local.solve(rhs2, trans='H')

            Hv += -omega**2 * np.real(u.conj() * lam2)

        return Hv

    # --------------------------------------------------
    # Compute spectral scale
    # --------------------------------------------------
    lambda_eff = g_total.dot(H0_matvec(g_total)) / \
                 (g_total.dot(g_total) + 1e-16)

    print("Effective eigenvalue scale:", lambda_eff)

    # --------------------------------------------------
    # L-curve computation
    # --------------------------------------------------
    phi = []
    psi = []

    for beta in beta_values:

        beta_physical = beta * lambda_eff

        def H_matvec(v):

            Hv = H0_matvec(v)
            Hv += beta_physical * v
            return Hv

        H_linop = spla.LinearOperator((n, n), matvec=H_matvec)

        dm, _ = spla.cg(H_linop, -g_total, maxiter=cg_maxiter)

        phi.append(np.linalg.norm(g_total + H_matvec(dm)))
        psi.append(np.linalg.norm(dm))

    phi = np.log(np.array(phi) + 1e-16)
    psi = np.log(np.array(psi) + 1e-16)

    # --------------------------------------------------
    # Curvature computation (stable)
    # --------------------------------------------------
    dphi = np.gradient(phi)
    dpsi = np.gradient(psi)
    ddphi = np.gradient(dphi)
    ddpsi = np.gradient(dpsi)

    eps = 1e-12
    den = (dphi**2 + dpsi**2)**1.5

    curvature = np.abs(dphi * ddpsi - dpsi * ddphi) / (den + eps)

    curvature[np.isnan(curvature)] = 0
    curvature[np.isinf(curvature)] = 0

    idx_best = np.argmax(curvature)

    return beta_values[idx_best], phi, psi, curvature

In [14]:
import numpy as np
import scipy.sparse.linalg as spla

def Hessian(m, f,
            d_obs,
            P, rec_idx,
            src_x, src_z,
            nx, nz, n, idx,
            dx, dz, npml, vmax,
            alpha):

    omega = 2 * np.pi * f
    alpha_scaled = alpha / (f**2)

    H_data = []

    # Build forward operator
    A = Forward_modelling.helmholtz5(m, omega, dx, dz, npml, vmax)
    LU = spla.splu(A.tocsc())

    # --- Forward wavefields for all shots ---
    for isrc, s in enumerate(src_x):

        q = np.zeros(n, dtype=complex)
        q[idx(src_z, s)] = 1.0

        u = LU.solve(q)

        # Store wavefield + LU for Hessian construction
        H_data.append((u, LU))

    # --- Define Hessian-vector product ---
    def H_matvec(v):

        Hv = np.zeros(n)

        for (u, LU_local) in H_data:

            # Forward incremental wavefield
            w = LU_local.solve(-omega**2 * (v * u))

            # Project to receivers
            Jv = P @ w

            # Adjoint solve
            rhs2 = np.zeros(n, dtype=complex)
            rhs2[rec_idx] = Jv
            lam2 = LU_local.solve(rhs2, trans='H')

            Hv += -omega**2 * np.real(u.conj() * lam2)

        Hv += alpha_scaled * v
        return Hv

    H_linop = spla.LinearOperator((n, n), matvec=H_matvec)

    return H_linop

In [15]:
def Full_newton(c_init, frequencies, d_obs,
                  P, rec_idx,
                src_x, src_z,
                nx, nz, n, idx,
                dx,dz,npml,vmax,
                beta0,
                n_iterations,
                minres_maxiter):

    m = 1/c_init**2
    misfits = []

    for f in frequencies:

        print("\n===== Frequency:", f, "Hz =====")
        omega = 2*np.pi*f
        beta = beta0

        for iteration in range(n_iterations):

            g = np.zeros(n)
            H_data = []
            misfit = 0.0

            # ----- Build operator -----
            A = Forward_modelling.helmholtz5(m,omega,dx,dz,npml,vmax)
            LU = spla.splu(A.tocsc())

            # ----- Forward & Adjoint -----
            for isrc, s in enumerate(src_x):

                q = np.zeros(n, dtype=complex)
                q[idx(src_z, s)] = 1.0

                u = LU.solve(q)
                r = P @ u - d_obs[f][isrc]
                misfit += 0.5 * np.linalg.norm(r)**2

                rhs = np.zeros(n, dtype=complex)
                rhs[rec_idx] = r
                lam = LU.solve(rhs, trans='H')

                g += -omega**2 * np.real(u.conj() * lam)

                # Store for full Hessian
                H_data.append((u, lam, LU))

            misfits.append(misfit)
            print("Iter:", iteration+1, "Misfit:", misfit, "beta:", beta)

            # ----- Full Newton Hessian-vector product -----
            def H_matvec(v):

                Hv = np.zeros(n)

                for (u, lam, LU_local) in H_data:

                    # First variation of wavefield
                    w = LU_local.solve(-omega**2 * (v * u))
                    Jv = P @ w

                    # Adjoint of Jv
                    rhs2 = np.zeros(n, dtype=complex)
                    rhs2[rec_idx] = Jv
                    lam2 = LU_local.solve(rhs2, trans='H')

                    term1 = -omega**2 * np.real(u.conj() * lam2)
                    term2 = -omega**2 * np.real(w.conj() * lam)

                    Hv += term1 + term2

                Hv += beta * v
                return Hv

            H_linop = spla.LinearOperator((n, n), matvec=H_matvec)

            # Solve (H + βI) dm = -g   (indefinite-safe)
            dm, info = spla.minres(H_linop, -g, maxiter=minres_maxiter)

            # ----- Trial step -----
            m_trial = m + dm.reshape(nz, nx)

            A_trial = Forward_modelling.helmholtz5(m_trial,omega,dx,dz,npml,vmax)
            LU_trial = spla.splu(A_trial.tocsc())

            misfit_trial = 0.0
            for isrc, s in enumerate(src_x):

                q = np.zeros(n, dtype=complex)
                q[idx(src_z, s)] = 1.0

                u_trial = LU_trial.solve(q)
                r_trial = P @ u_trial - d_obs[f][isrc]
                misfit_trial += 0.5 * np.linalg.norm(r_trial)**2

            # ----- Adaptive Levenberg–Marquardt damping -----
            if misfit_trial < misfit:
                m = m_trial
                beta *= 0.5
            else:
                beta *= 5.0

            # Stabilization
            m = gaussian_filter(m, sigma=1)

            # Physical bounds (slowness squared)
            m = np.clip(m, 1/2600**2, 1/1500**2)

    return m, misfits


In [16]:
# def Full_newton(c_init, frequencies, d_obs,
#                   P, rec_idx,
#                 src_x, src_z,
#                 nx, nz, n, idx,
#                 dx,dz,npml,vmax,
#                 beta0,
#                 n_iterations,
#                 minres_maxiter):

#     import numpy as np
#     import scipy.sparse.linalg as spla

#     m = 1 / c_init**2
#     misfits = []

#     for f in frequencies:

#         print("\n===== Frequency:", f, "Hz =====")
#         omega = 2*np.pi*f

#         for iteration in range(n_iterations):

#             g = np.zeros(n)
#             H_data = []
#             misfit = 0.0

#             # ----- Build Helmholtz -----
#             A = Forward_modelling.helmholtz5(m,omega,dx,dz,npml,vmax)
#             LU = spla.splu(A.tocsc())

#             # ----- Forward & Adjoint -----
#             for isrc, s in enumerate(src_x):

#                 q = np.zeros(n, dtype=complex)
#                 q[idx(src_z, s)] = 1.0

#                 u = LU.solve(q)
#                 r = P @ u - d_obs[f][isrc]
#                 misfit += 0.5 * np.linalg.norm(r)**2

#                 rhs = np.zeros(n, dtype=complex)
#                 rhs[rec_idx] = r
#                 lam = LU.solve(rhs, trans='H')

#                 g += -omega**2 * np.real(u.conj() * lam)
#                 H_data.append((u, lam, LU))

#             misfits.append(misfit)
#             print("Iter:", iteration+1, "Misfit:", misfit)

#             g_norm = np.linalg.norm(g)

#             # ---- Small adaptive damping (vanishes near solution) ----
#             beta = beta0 * g_norm

#             # ----- Full Hessian-vector product -----
#             def H_matvec(v):

#                 Hv = np.zeros(n)

#                 for (u, lam, LU_local) in H_data:

#                     # First wavefield variation
#                     w = LU_local.solve(-omega**2 * (v * u))
#                     Jv = P @ w

#                     rhs2 = np.zeros(n, dtype=complex)
#                     rhs2[rec_idx] = Jv
#                     lam2 = LU_local.solve(rhs2, trans='H')

#                     term1 = -omega**2 * np.real(u.conj() * lam2)
#                     term2 = -omega**2 * np.real(w.conj() * lam)

#                     Hv += term1 + term2

#                 Hv += beta * v
#                 return Hv

#             H_linop = spla.LinearOperator((n, n), matvec=H_matvec)

#             # ----- Accurate Newton solve -----
#             dm, info = spla.minres(
#                 H_linop,
#                 -g,
#                 maxiter=minres_maxiter,
#                 rtol=1e-6
#             )

#             if info != 0:
#                 print("Warning: MINRES not fully converged")

#             # ----- Armijo line search -----
#             alpha = 1.0
#             c = 1e-4

#             while True:

#                 m_trial = m + alpha * dm.reshape(nz, nx)

#                 A_trial = Forward_modelling.helmholtz5(m_trial,omega,dx,dz,npml,vmax)
#                 LU_trial = spla.splu(A_trial.tocsc())

#                 misfit_trial = 0.0
#                 for isrc, s in enumerate(src_x):

#                     q = np.zeros(n, dtype=complex)
#                     q[idx(src_z, s)] = 1.0

#                     u_trial = LU_trial.solve(q)
#                     r_trial = P @ u_trial - d_obs[f][isrc]
#                     misfit_trial += 0.5 * np.linalg.norm(r_trial)**2

#                 if misfit_trial <= misfit + c * alpha * np.dot(g.flatten(), dm):
#                     break

#                 alpha *= 0.5
#                 if alpha < 1e-6:
#                     print("Line search failed")
#                     break

#             m = m_trial

#             # ----- Physical bounds -----
#             m = np.clip(m, 1/2600**2, 1/1500**2)

#             # ---- Check quadratic regime ----
#             if g_norm < 1e-4:
#                 print("Entering asymptotic Newton regime")

#     return m, misfits